# CLaRa: Continuous Latent Reasoning — Full-Paper Reproduction (Kaggle T4/P100)

> **Paper:** *CLaRa: Bridging Retrieval and Generation with Continuous Latent Reasoning*  
> He et al., Apple / University of Edinburgh — February 2026  
> **Code:** https://github.com/apple/ml-clara

---

## Overview

CLaRa is an end-to-end Retrieval-Augmented Generation (RAG) framework that addresses two key limitations of classical RAG:

1. **Efficiency** — Documents are compressed into compact *memory tokens* (16×–256× compression) once and reused for both retrieval and generation, eliminating redundant text processing.
2. **Optimization** — Retrieval and generation are jointly trained via a differentiable top-k selection (Straight-Through estimator), allowing generator gradients to directly update the retriever.

### Two-Stage Training Pipeline

| Stage | Name | What trains | Loss | Purpose |
|-------|------|-------------|------|---------|
| **Stage I** | SCP — Salient Compressor Pretraining | `compressor` + `generator` LoRA | $\mathcal{L}_{CE} + \lambda \cdot \mathcal{L}_{MSE}$ | Learn to compress documents into memory tokens that preserve salient semantics |
| **Stage II** | E2E — End-to-End Joint Training | `query` + `generator` LoRA | $\mathcal{L}_{NTP}$ (next-token prediction) | Differentiable top-k retrieval + joint optimisation via ST estimator |

### This Notebook: Two Experimental Flows

```
Flow 1 ─── Train from Scratch (HotpotQA)
           Stage I  →  Stage II  →  Evaluate (HotpotQA)
           Purpose: Reproduce the full paper pipeline end-to-end.

Flow 2 ─── Transfer Learning from Apple Pretrained Weights
           Download E2E checkpoint → Convert → Fine-tune Stage II
           Evaluate on:
             (a) Apple pretrained model   (zero-shot, no fine-tuning)
             (b) Fine-tuned model         (TriviaQA + SQuAD separately)
           Purpose: Leverage Apple's SCP pretraining; demonstrate transfer.
```

### Paper Hyperparameters (Appendix B.4, Table 10)

| Hyperparameter | Paper Value | This Notebook |
|----------------|-------------|---------------|
| Base model | Mistral-7B-Instruct-v0.2 | Same |
| LoRA rank (r) | 16 | Same |
| LoRA alpha | 32 | Same |
| LoRA dropout | 0.1 | Same |
| Stage I LR | 2e-4 | Same |
| Stage II LR | 5e-6 | Same |
| Compression ratio | 16× (n_memory=16, doc_len=256) | Same |
| top-k documents | 5 | 2 (T4 VRAM limit) |
| Candidates | 20 | 8 (T4 VRAM limit) |
| Epochs | 1 | 1 |
| Warmup ratio | 0.03 | Same |
| ST temperature τ | — | 0.7 |

---

**Note on T4 adaptations:** The paper trains on 8×H100 GPUs. On a single T4 (16 GB), we reduce `num_candidates=8` and `top_k=2`. All other architectural choices are faithful to the paper.

---
##  Section 0 — Environment Setup

**Run once.** Clone the repo and install dependencies, then **restart the kernel** before running any further cells.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0-A  │  Clone repository & install dependencies
# Run ONCE. After this cell completes → Session > Restart & Run All (skip cell 0-A).
# ═══════════════════════════════════════════════════════════════════════════════

import subprocess, sys, os

REPO_URL  = "https://github.com/Duy-Tuyen/introml-clara-implementation.git"
REPO_NAME = "introml-clara-implementation"
REPO_ROOT = f"/kaggle/working/{REPO_NAME}"


def _run(label: str, cmd: list) -> None:
    """Run subprocess, capture output, print ONCE — tránh Papermill duplicate."""
    print(label, flush=True)
    r = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,   # ← capture, không để leak ra 2 streams
        stderr=subprocess.STDOUT, # ← gộp stderr vào stdout
        text=True,
    )
    # Lọc bỏ các dòng không cần thiết (debugger warnings, progress bars)
    skip_prefixes = (
        "0.00s - Debugger",
        "0.00s - make the debugger",
        "0.00s - to python",
        "0.00s - Note: Debugging",
        "━━━",  # pip progress bars
    )
    lines = [
        l for l in r.stdout.splitlines()
        if not any(l.strip().startswith(p) for p in skip_prefixes)
    ]
    if lines:
        print('\n'.join(lines), flush=True)
    if r.returncode != 0:
        raise subprocess.CalledProcessError(r.returncode, cmd)


# 1. Clone
subprocess.run(["rm", "-rf", REPO_ROOT],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
_run("[1/3] Cloning repository...",
     ["git", "clone", "-b", "feature/hieu2", REPO_URL, REPO_ROOT])

# 2. Install dependencies (pip output captured → không duplicate)
_run("[2/3] Installing dependencies...",
     [sys.executable, "-m", "pip", "install",
      "-r", f"{REPO_ROOT}/requirements.txt", "-q"])

# 3. Patch môi trường (import trực tiếp — không subprocess → không duplicate)
print("[3/3] Patching Kaggle environment...", flush=True)
sys.path.insert(0, REPO_ROOT)
from setup_env import setup_kaggle_env
setup_kaggle_env(verbose=True)

print("\n Setup complete. Please RESTART the kernel before continuing.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0-B  │  Working directory & path configuration
# Run this cell FIRST after every kernel restart.
# ═══════════════════════════════════════════════════════════════════════════════

import os, sys

REPO_ROOT = "/kaggle/working/introml-clara-implementation"
assert os.path.isdir(REPO_ROOT), (
    f"Repository not found at {REPO_ROOT}. "
    "Please run Cell 0-A first, then restart the kernel."
)

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"Working directory : {os.getcwd()}")
print(f"Python path entry : {sys.path[0]}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0-C  │  GPU / VRAM diagnostics
# ═══════════════════════════════════════════════════════════════════════════════

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable GPU: Settings > Accelerator > T4 or P100.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU   : {gpu_name}")
print(f"VRAM  : {vram_gb:.1f} GB")
print(f"CUDA  : {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

if vram_gb < 14:
    print("\n Warning: Less than 14 GB VRAM. Consider reducing doc_max_length or batch_size.")
else:
    print("\n VRAM looks sufficient for T4-friendly config (num_candidates=8, top_k=2).")

---
## Section 1 
### Flow 1 — Train from Scratch on HotpotQA

Full two-stage training pipeline on HotpotQA (multi-hop QA benchmark).

**Why HotpotQA instead of the paper's Qwen-32B synthetic data?**  
The paper synthesises SCP pretraining data using a locally-deployed Qwen-32B model on 2M Wikipedia documents — infeasible on a single T4. HotpotQA provides real QA supervision with multi-hop context, making it a practical substitute.

```
HotpotQA training set: 90,185 samples (Table 9 in paper)
HotpotQA eval set   :  7,384 samples
```

### Stage I — Salient Compressor Pretraining (SCP)

Trains `compressor` + `generator` LoRA adapters.  
Objective: $\mathcal{L}_{SCP} = \mathcal{L}_{CE} + \lambda \cdot \mathcal{L}_{MSE}$  
The MSE term aligns memory token representations with document hidden states, enforcing semantic fidelity.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2  │  Flow 1 — Stage I: Salient Compressor Pretraining (SCP)
#
# Paper §2.2: "Only the compressor LoRA θ_c is trained via cross-entropy loss.
# The generator is jointly trained to produce answers from memory tokens + query."
# Loss: L_CE + λ * L_MSE  (λ = 0.1, paper Appendix B.4)
# LR: 2e-4  |  Epochs: 1  |  Warmup: 3%
# ═══════════════════════════════════════════════════════════════════════════════

import os, sys, gc, torch
sys.path.insert(0, "/kaggle/working/introml-clara-implementation")
os.chdir("/kaggle/working/introml-clara-implementation")

from configs.config import CLaRaConfig
from models.clara_model import build_clara_model
from data.dataset import get_dataloaders
from scripts.train_stage1 import train_stage1

FLOW1_CKPT = "/kaggle/working/clara-ckpts-flow1"

print("╔" + "═" * 60 + "╗")
print("║  FLOW 1 — Stage I: Salient Compressor Pretraining (SCP)    ║")
print("║  Dataset: HotpotQA | Train: 500 | Val: 100                ║")
print("╚" + "═" * 60 + "╝") 

cfg = CLaRaConfig(dataset_name='hotpotqa', n_train=500, n_val=100,
                   output_dir=FLOW1_CKPT)
model, tokenizer = build_clara_model(cfg)
train_dl, val_dl = get_dataloaders(tokenizer, cfg)
train_stage1(model, train_dl, val_dl, cfg)

# Giải phóng VRAM trước khi sang Stage II
del model; gc.collect(); torch.cuda.empty_cache()
print(f"\n Stage I complete → {FLOW1_CKPT}/stage1_ep1")


### Stage II — End-to-End Joint Training

Initialises the `query` adapter from Stage I compressor weights, then jointly trains `query` + `generator` via differentiable top-k retrieval.

**Key mechanism (paper §3):** The Straight-Through (ST) estimator enables gradient flow from the generator loss back through the discrete top-k selection to the query encoder — the core innovation of CLaRa.

$Z = Z_{\text{hard}} + (Z_{\text{soft}} - \text{SG}(Z_{\text{soft}}))$

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3  │  Flow 1 — Stage II: End-to-End Joint Training
#
# Paper §3: Joint training of query reasoner θ_qr and generator θ_g
# via NTP loss with differentiable top-k selection (ST estimator).
# Query adapter initialised from compressor weights (paper §3, init strategy).
# LR: 5e-6  |  Epochs: 1  |  top_k=2 (paper uses 5, reduced for T4)
# ═══════════════════════════════════════════════════════════════════════════════

import os, sys, gc, torch
sys.path.insert(0, "/kaggle/working/introml-clara-implementation")
os.chdir("/kaggle/working/introml-clara-implementation")

from configs.config import CLaRaConfig
from models.clara_model import build_clara_model
from data.dataset import get_retrieval_dataloaders
from scripts.train_stage2 import train_stage2, _load_stage1

FLOW1_CKPT       = "/kaggle/working/clara-ckpts-flow1"
FLOW1_STAGE1_DIR = f"{FLOW1_CKPT}/stage1_ep1"

assert os.path.isdir(FLOW1_STAGE1_DIR), f"Stage I checkpoint không tìm thấy: {FLOW1_STAGE1_DIR}"

print("╔" + "═" * 60 + "╗")
print("║  FLOW 1 — Stage II: End-to-End Joint Training               ║")
print("║  Dataset: HotpotQA | ST estimator τ=0.7 | top_k=2          ║")
print("╚" + "═" * 60 + "╝")

cfg = CLaRaConfig(dataset_name='hotpotqa', n_train=500, n_val=100,
                  output_dir=FLOW1_CKPT,
                  stage1_ckpt_dir=FLOW1_STAGE1_DIR)
model, tokenizer = build_clara_model(cfg)
_load_stage1(model, FLOW1_STAGE1_DIR)
train_dl, val_dl = get_retrieval_dataloaders(tokenizer, cfg)
train_stage2(model, train_dl, val_dl, cfg)

del model; gc.collect(); torch.cuda.empty_cache()
print(f"\n✅ Stage II complete → {FLOW1_CKPT}/stage2_ep1")


### Evaluation — Flow 1 Model (HotpotQA)

Evaluates the from-scratch model using SQuAD-style EM and F1.  
Paper reference (Oracle setting, CLaRa-Mistral-7B 16×): **EM ≈ 56.76%, F1 ≈ 69.57%** on HotpotQA (Table 2, instruction-tuned init).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4  │  Flow 1 — Evaluate model trained from scratch on HotpotQA
#
# Metrics (paper B.3):
#   EM  — Exact Match: % predictions exactly matching gold answer
#         (after SQuAD normalisation: lowercase, strip punct/articles)
#   F1  — Token-level F1: harmonic mean of precision and recall
# ═══════════════════════════════════════════════════════════════════════════════

import os, sys, gc, torch
sys.path.insert(0, "/kaggle/working/introml-clara-implementation")
os.chdir("/kaggle/working/introml-clara-implementation")

from configs.config import CLaRaConfig
from models.clara_model import build_clara_model
from data.dataset import get_retrieval_eval_loader
from scripts.evaluate import evaluate, load_checkpoint

FLOW1_CKPT       = "/kaggle/working/clara-ckpts-flow1"
FLOW1_STAGE2_DIR = f"{FLOW1_CKPT}/stage2_ep1"

assert os.path.isdir(FLOW1_STAGE2_DIR), f"Stage II checkpoint không tìm thấy: {FLOW1_STAGE2_DIR}"

print("╔" + "═" * 60 + "╗")
print("║  FLOW 1 — Evaluation: HotpotQA  (oracle mode)              ║")
print("║  Paper ref: EM~56.8%  F1~69.6%  [Table 2]                  ║")
print("╚" + "═" * 60 + "╝")

cfg = CLaRaConfig(dataset_name='hotpotqa', eval_mode='oracle',
                  eval_batch_size=4, n_val=500)
model, tokenizer = build_clara_model(cfg)
load_checkpoint(model, FLOW1_STAGE2_DIR)
val_loader = get_retrieval_eval_loader(tokenizer, cfg, split='validation')
results = evaluate(model, val_loader, cfg)

print(f"\n  Exact Match : {results['em']*100:.2f}%")
print(f"  F1 Score    : {results['f1']*100:.2f}%")
print(f"  Samples     : {results['n_samples']}")

# Lưu CSV
import csv
from datetime import datetime
os.makedirs('results', exist_ok=True)
csv_file = 'results/eval_scores.csv'
file_exists = os.path.isfile(csv_file)
with open(csv_file, 'a', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(['Timestamp','Model_Version','Dataset','Eval_Mode','Exact_Match(%)','F1_Score(%)'])
    writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                     'Flow1_ScratchTrained_HotpotQA', 'hotpotqa', 'oracle',
                     f"{results['em']*100:.2f}", f"{results['f1']*100:.2f}"])

del model; gc.collect(); torch.cuda.empty_cache()
print(f"\n Kết quả đã lưu vào: {csv_file}")

---
##  Section 2 — Results Summary

Aggregates all evaluation results from `results/eval_scores.csv` and displays a formatted comparison table.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5  │  Results summary — all evaluation runs
# ═══════════════════════════════════════════════════════════════════════════════

import os
import pandas as pd

CSV_PATH = "results/eval_scores.csv"

if not os.path.exists(CSV_PATH):
    print(f"No results found at {CSV_PATH}. Run evaluation cells first.")
else:
    df = pd.read_csv(CSV_PATH)

    print("═" * 80)
    print("  CLaRa EXPERIMENT RESULTS SUMMARY")
    print("═" * 80)
    print()

    # Format display
    display_df = df[[
        'Model_Version', 'Dataset', 'Eval_Mode',
        'Exact_Match(%)', 'F1_Score(%)', 'Timestamp'
    ]].copy()

    # Sort for readability
    display_df = display_df.sort_values(['Dataset', 'Model_Version'])

    pd.set_option('display.max_colwidth', 45)
    pd.set_option('display.width', 120)
    print(display_df.to_string(index=False))

    print()
    print("─" * 80)
    print("PAPER REFERENCE (Table 2 — Oracle, CLaRa-Mistral-7B 16×):")
    print("  NQ        : EM=63.29%  F1=71.54%")
    print("  HotpotQA  : EM=57.54%  F1=71.17%")
    print("  (Instruction-tuned init, Normal setting)")
    print("─" * 80)
    print()
    print("NOTE: Our results are expected to be lower due to:")
    print("  • Single T4 GPU (paper: 8×H100)")
    print("  • Reduced num_candidates=8 vs 20 in paper")
    print("  • top_k=2 vs 5 in paper")
    print("  • HotpotQA substitute for Qwen-32B synthetic SCP data")
    print("  • 8,000 training samples vs full dataset in paper")

---
##  Section 3 — Qualitative Inference Examples

Runs interactive inference to inspect the model's predictions qualitatively.  
Change `TEST_CASES` or `CKPT_TO_USE` to test different models.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6  │  Qualitative inference — inspect model predictions
# ═══════════════════════════════════════════════════════════════════════════════

import os, sys, torch
from peft import load_peft_weights_local, set_peft_model_state_dict
from configs.config import CLaRaConfig
from models.clara_model import build_clara_model

# ── Configure which checkpoint to use ────────────────────────────────────────
# Options:
#   Flow 1 model     : "/kaggle/working/clara-ckpts-flow1/stage2_ep1"
#   Apple pretrained : "/kaggle/working/pretrained-apple-e2e-converted"
#   Fine-tuned TriviaQA: "/kaggle/working/clara-ckpts-ft-triviaqa/stage2_ep1"
#   Fine-tuned SQuAD   : "/kaggle/working/clara-ckpts-ft-squad/stage2_ep1"

CKPT_TO_USE = "/kaggle/working/clara-ckpts-ft-triviaqa/stage2_ep1"

TEST_CASES = [
    {
        'doc': 'The Battle of Hastings was fought on 14 October 1066 between '
               'the Norman-French army of William, the Duke of Normandy, and '
               'an English army under the Anglo-Saxon King Harold Godwinson.',
        'q'  : 'When was the Battle of Hastings fought?',
        'expected': '14 October 1066',
    },
    {
        'doc': 'Weldenia is a monotypic genus of flowering plants in the family '
               'Commelinaceae, native to Mexico and Guatemala.',
        'q'  : 'Which genus grows originally in Mexico and Guatemala, '
               'Phylica or Weldenia?',
        'expected': 'Weldenia',
    },
    {
        'doc': 'Albert Einstein was born on 14 March 1879 in Ulm, in the '
               'Kingdom of Württemberg in the German Empire. He developed the '
               'theory of relativity.',
        'q'  : 'Where was Albert Einstein born?',
        'expected': 'Ulm',
    },
]

# ── Load model ────────────────────────────────────────────────────────────────
print(f"Loading checkpoint: {CKPT_TO_USE}")
assert os.path.isdir(CKPT_TO_USE), f"Checkpoint not found: {CKPT_TO_USE}"

cfg = CLaRaConfig()
model, tokenizer = build_clara_model(cfg)

query_dir = os.path.join(CKPT_TO_USE, 'adapters', 'query')
gen_dir   = os.path.join(CKPT_TO_USE, 'adapters', 'generator')
extra_pth = os.path.join(CKPT_TO_USE, 'clara_stage2_extra.pth')

if os.path.isdir(query_dir):
    set_peft_model_state_dict(model.backbone,
                              load_peft_weights_local(query_dir), adapter_name='query')
if os.path.isdir(gen_dir):
    set_peft_model_state_dict(model.backbone,
                              load_peft_weights_local(gen_dir), adapter_name='generator')
if os.path.exists(extra_pth):
    model.mem_token_embed.data = torch.load(extra_pth, map_location='cuda')['mem_token_embed']

print("✓ Checkpoint loaded.\n")

# ── Run inference ─────────────────────────────────────────────────────────────
model.eval()
print("═" * 60)
print("INFERENCE EXAMPLES")
print("═" * 60)

with torch.no_grad():
    for i, t in enumerate(TEST_CASES, 1):
        doc_enc = tokenizer(
            [t['doc']], max_length=cfg.doc_max_length,
            padding='max_length', truncation=True, return_tensors='pt')
        q_enc = tokenizer(
            f"[INST] {t['q']} [/INST]", max_length=cfg.max_qa_len,
            padding='max_length', truncation=True, return_tensors='pt')

        doc_ids  = doc_enc['input_ids'].unsqueeze(0).cuda()
        doc_mask = doc_enc['attention_mask'].unsqueeze(0).cuda()
        cand_mask = torch.ones(1, 1, dtype=torch.long, device='cuda')
        q_ids    = q_enc['input_ids'].cuda()
        q_mask   = q_enc['attention_mask'].cuda()

        answer = model.generate_answer_e2e(
            doc_ids, doc_mask, cand_mask, q_ids, q_mask,
            max_new_tokens=32
        )[0]

        print(f"\n[{i}] Question : {t['q']}")
        print(f"    Expected : {t['expected']}")
        print(f"    Model    : {answer}")
        match = t['expected'].lower() in answer.lower()
        print(f"    Match    : {'✓ YES' if match else '✗ NO'}")

print("\n" + "═" * 60)

---
##  Section 4 — Experimental Notes & Reproducibility

### T4 GPU Adaptations vs. Paper

| Setting | Paper | This notebook | Reason |
|---------|-------|---------------|--------|
| `num_candidates` | 20 | 8 | T4 VRAM |
| `top_k` | 5 | 2 | T4 VRAM |
| SCP pretraining data | 2M Wiki docs via Qwen-32B | HotpotQA (8k samples) | No Qwen-32B access |
| Training GPUs | 8×H100 | 1×T4 | Kaggle free tier |
| Training samples | Full dataset | 8,000 | Runtime constraint |

### Flow 1 vs Flow 2 — Key Differences

| | Flow 1 (Scratch) | Flow 2 (Transfer) |
|-|------------------|-------------------|
| Stage I compressor | Trained on HotpotQA | Apple's (2M docs, Qwen-32B) |
| Stage II init | Flow 1 Stage I | Apple E2E converted |
| Fine-tune datasets | HotpotQA | TriviaQA, SQuAD |
| Expected quality | Lower (weak SCP) | Higher (strong SCP) |

### Expected Results (Oracle setting, 16× compression)

From paper Table 2 (instruction-tuned init, closest to Flow 2):

| Dataset | EM | F1 |
|---------|----|----|  
| NQ | 63.29% | 71.54% |
| HotpotQA | 57.54% | 71.17% |

Our results will be lower due to the T4 adaptations above. This is expected and documented.

### Citation

```bibtex
@article{he2026clara,
  title   = {CLaRa: Bridging Retrieval and Generation with Continuous Latent Reasoning},
  author  = {He, Jie and Bai, Richard He and Williamson, Sinead and Pan, Jeff Z. 
             and Jaitly, Navdeep and Zhang, Yizhe},
  journal = {arXiv preprint arXiv:2511.18659},
  year    = {2026}
}
```